# NiyamTrace-X Wave 3 / Experiment 12 — τ³-Bench Stateful Policy Transfer & Multi-Model Evaluation

This notebook runs the current Sierra Research τ-bench codebase in its **τ³-era v1.0.1+ form**, covering realistic stateful customer-service domains. It uses the benchmark's official simulator/reward and then performs a separate NiyamTrace-X-oriented effect audit over the saved trajectories.

### What is measured
- Official task success/reward by model and domain.
- Read vs. write tool-call profiles.
- Duplicate/repeated write effects and argument drift within trajectories.
- Multi-step irreversible-effect frontier proxies.
- Cross-domain and cross-model safety/utility trade-offs.
- Exact repository commit and CLI help are archived.

The post-hoc effect audit is clearly labeled a **transfer proxy**; it does not replace τ³-Bench's official reward.


In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','uv','pandas','numpy','matplotlib'])


In [ ]:
from pathlib import Path
import os, sys, json, re, math, time, random, hashlib, zipfile, shutil, subprocess, statistics, tempfile, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=20260911
random.seed(SEED); np.random.seed(SEED)
BASE=Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS=BASE/'niyamtrace_q1_wave3_results'
RESULTS.mkdir(parents=True,exist_ok=True)
print('BASE:',BASE)
print('RESULTS:',RESULTS)


In [ ]:
# Multi-model configuration. For a formal run, use at least 3 independent model families.
# Preferred: set NTX_MODELS_JSON in the environment so credentials never enter the notebook.
# Format:
# [
#   {"label":"qwen35-122b","model":"Qwen/Qwen3.5-122B-A10B-FP8","api_url":"http://HOST:PORT/v1","api_key":"EMPTY"},
#   {"label":"gpt-oss-120b","model":"openai/gpt-oss-120b","api_url":"http://HOST:PORT/v1","api_key":"EMPTY"}
# ]
raw=os.getenv('NTX_MODELS_JSON','').strip()
MODELS=json.loads(raw) if raw else []
for m in MODELS:
    for k in ['label','model','api_url']:
        if not m.get(k): raise ValueError(f'Model spec missing {k}: {m}')
    m.setdefault('api_key','EMPTY')
print('Configured model endpoints:', [m['label'] for m in MODELS])
if not MODELS:
    print('No model endpoints configured. Benchmark discovery/mapping cells can still run; model inference cells will record SKIPPED_NO_MODEL_CONFIG rather than invent results.')


In [ ]:
MODE=os.getenv('NTX_RUN_MODE','QUICK').upper(); assert MODE in {'QUICK','STANDARD','FULL'}
DOMAINS=['airline','retail','telecom','banking_knowledge']
NUM_TASKS={'QUICK':5,'STANDARD':25,'FULL':None}[MODE]
TRIALS=int(os.getenv('TAU_TRIALS','1'))
TAU=BASE/'tau2-bench'
if not TAU.exists():
    p=subprocess.run(['git','clone','-q','https://github.com/sierra-research/tau2-bench.git',str(TAU)],capture_output=True,text=True)
    if p.returncode: print('Clone failed:',p.stderr)
if TAU.exists():
    # Prefer the corrected v1.0.1 release if available; otherwise record current HEAD.
    tags=subprocess.run(['git','-C',str(TAU),'tag'],capture_output=True,text=True).stdout.split()
    if 'v1.0.1' in tags: subprocess.run(['git','-C',str(TAU),'checkout','-q','v1.0.1'])
    commit=subprocess.run(['git','-C',str(TAU),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
else: commit='UNAVAILABLE'
print('tau repo commit:',commit)


In [ ]:
# Install τ-bench in its own uv-managed environment. This preserves the notebook kernel.
install={'status':'SKIPPED_REPO_UNAVAILABLE'}
if TAU.exists():
    p=subprocess.run(['uv','sync'],cwd=TAU,capture_output=True,text=True)
    install={'status':'OK' if p.returncode==0 else 'ERROR','returncode':p.returncode,'stdout_tail':p.stdout[-2000:],'stderr_tail':p.stderr[-2000:]}
    h=subprocess.run(['uv','run','tau2','--help'],cwd=TAU,capture_output=True,text=True)
    (RESULTS/'exp12_tau_cli_help.txt').write_text(h.stdout+'\nSTDERR\n'+h.stderr)
(RESULTS/'exp12_install_status.json').write_text(json.dumps(install,indent=2)); print(install)


In [ ]:
# Official multi-model runs. By default each endpoint is exposed to LiteLLM as an OpenAI-compatible model.
# Optional per-model field `tau_model` can override the CLI model string.
run_rows=[]
user_model=os.getenv('TAU_USER_MODEL','').strip()
for m in MODELS:
    for domain in DOMAINS:
        before=set(TAU.rglob('*.json')) if TAU.exists() else set()
        env=os.environ.copy(); env['OPENAI_API_KEY']=m.get('api_key','EMPTY'); env['OPENAI_API_BASE']=m['api_url']; env['OPENAI_BASE_URL']=m['api_url']
        agent_model=m.get('tau_model',f"openai/{m['model']}")
        u_model=user_model or agent_model
        cmd=['uv','run','tau2','run','--domain',domain,'--agent-llm',agent_model,'--user-llm',u_model,'--num-trials',str(TRIALS),'--task-split','base','--max-concurrency','1']
        if NUM_TASKS is not None: cmd += ['--num-tasks',str(NUM_TASKS)]
        t=time.time(); p=subprocess.run(cmd,cwd=TAU,env=env,capture_output=True,text=True) if TAU.exists() else None
        after=set(TAU.rglob('*.json')) if TAU.exists() else set(); new=sorted(str(x) for x in after-before)
        out=RESULTS/f"exp12_tau_{m['label']}_{domain}.log"; out.write_text((p.stdout if p else '')+'\nSTDERR\n'+(p.stderr if p else 'repo unavailable'))
        run_rows.append({'benchmark':'tau3','model':m['label'],'domain':domain,'returncode':p.returncode if p else None,'seconds':time.time()-t,'new_json_files':json.dumps(new),'log':str(out)})
if not MODELS: run_rows=[{'benchmark':'tau3','model':'NONE','domain':'ALL','returncode':None,'seconds':0,'new_json_files':'[]','log':'SKIPPED_NO_MODEL_CONFIG'}]
run_df=pd.DataFrame(run_rows); run_df.to_csv(RESULTS/'exp12_tau_run_status.csv',index=False); display(run_df)


In [ ]:
# Collect likely simulation/trajectory files and normalize their official reward plus tool calls.
WRITE_HINT=re.compile(r'(update|book|cancel|refund|transfer|send|create|delete|archive|change|schedule|pay|purchase|reserve)',re.I)
def calls(o):
    out=[]
    def v(x):
        if isinstance(x,dict):
            if isinstance(x.get('function'),dict) and x['function'].get('name'):
                f=x['function']; out.append((f['name'],json.dumps(f.get('arguments',{}),sort_keys=True,default=str)))
            elif x.get('tool_name'): out.append((str(x['tool_name']),json.dumps(x.get('tool_args',{}),sort_keys=True,default=str)))
            elif x.get('name') and any(k in x for k in ['arguments','args']): out.append((str(x['name']),json.dumps(x.get('arguments',x.get('args',{})),sort_keys=True,default=str)))
            for y in x.values():v(y)
        elif isinstance(x,list):
            for y in x:v(y)
    v(o); return out
def scalar(o,names):
    if isinstance(o,dict):
        for k,v in o.items():
            if k.lower() in names and isinstance(v,(int,float,bool,str)):return v
        for v in o.values():
            r=scalar(v,names)
            if r is not None:return r
    if isinstance(o,list):
        for v in o:
            r=scalar(v,names)
            if r is not None:return r
    return None
traj=[]
if TAU.exists():
    for p in TAU.rglob('*.json'):
        sp=str(p).lower()
        if not any(k in sp for k in ['simulation','trajectory','traj','result']):continue
        try:o=json.loads(p.read_text())
        except:continue
        cs=calls(o)
        if not cs:continue
        names=[x[0] for x in cs]; writes=[x for x in cs if WRITE_HINT.search(x[0])]
        repeated=len(writes)-len(set(writes))
        traj.append({'path':str(p.relative_to(TAU)),'official_reward':scalar(o,{'reward','score','success','pass'}),'call_count':len(cs),'write_call_count':len(writes),'repeated_write_count':max(0,repeated),'unique_write_tools':len(set(x[0] for x in writes)),'effect_frontier_proxy':bool(writes)})
traj_df=pd.DataFrame(traj); traj_df.to_csv(RESULTS/'exp12_tau_trajectory_effect_audit.csv',index=False)
display(traj_df.head(30))


In [ ]:
# Cross-domain result extraction from logs (fail-loud, no invented scores).
log_rows=[]
for r in run_rows:
    lp=Path(r.get('log',''))
    text=lp.read_text(errors='ignore') if lp.exists() else ''
    nums=[]
    for pat in [r'pass[^0-9]{0,10}([0-9]+(?:\.[0-9]+)?)',r'reward[^0-9-]{0,10}(-?[0-9]+(?:\.[0-9]+)?)',r'score[^0-9-]{0,10}(-?[0-9]+(?:\.[0-9]+)?)']:
        nums += [float(x) for x in re.findall(pat,text,re.I)]
    log_rows.append({**{k:r.get(k) for k in ['model','domain','returncode']},'reported_numeric_metrics':json.dumps(nums[:50]),'metric_count':len(nums)})
log_df=pd.DataFrame(log_rows); log_df.to_csv(RESULTS/'exp12_tau_log_metric_extract.csv',index=False); display(log_df)


In [ ]:
if len(traj_df):
    vals=traj_df[['call_count','write_call_count','repeated_write_count']].mean(); fig,ax=plt.subplots(figsize=(7,4)); vals.plot(kind='bar',ax=ax); ax.set_title('τ³ trajectory effect profile'); ax.set_ylabel('Mean count per parsed trajectory'); fig.tight_layout(); fig.savefig(RESULTS/'exp12_tau_effect_profile.png',dpi=220); plt.show()
manifest={'experiment':'NTX-Q1-12','mode':MODE,'tau_commit':commit,'models':[{k:v for k,v in m.items() if k!='api_key'} for m in MODELS],'domains':DOMAINS,'result_files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in RESULTS.glob('exp12_*') if p.is_file()}}
(RESULTS/'exp12_manifest.json').write_text(json.dumps(manifest,indent=2))


In [ ]:
# FINAL CELL — package every result from this experiment and download it.
PREFIX='exp12_'
ZIP_OUT=BASE/'NTX_Q1_12_TAU3_STATEFUL_TRANSFER_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob('*')):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p,arcname=str(p.relative_to(RESULTS)))
sha=hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:',ZIP_OUT)
print('SHA-256:',sha)
print('Size MiB:',round(ZIP_OUT.stat().st_size/1024**2,3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab. ZIP is available at',ZIP_OUT)
